# Scraping Data Ulasan Play Store (Indonesia)

Notebook ini mengambil data ulasan aplikasi Duolingo secara mandiri dari internet menggunakan `google-play-scraper`.

Target:
- Minimal 10.000 sampel
- Data disimpan ke CSV untuk dipakai pada notebook analisis sentimen

Sumber data:
- Google Play Store

In [1]:
%pip install -q google-play-scraper pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google_play_scraper import Sort, reviews

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Scrapping review aplikasi Duolingo
APP_ID = "com.levelinfinite.sgameGlobal"
APP_NAME = "Honor of King"
LANG = "id"
COUNTRY = "id"
TARGET_COUNT = 100000
MAX_BATCH = 2000

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "reviews_hok.csv"

print(f"Target total sampel: ~{TARGET_COUNT}".replace(",", "."))

Target total sampel: ~100000


c:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def scrape_reviews_for_app(app_id: str, app_name: str, target_count: int = 2000):
    """Scrape review sampai target terpenuhi atau data habis."""
    all_rows = []
    token = None

    while len(all_rows) < target_count:
        batch_count = min(MAX_BATCH, target_count - len(all_rows))
        result, token = reviews(
            app_id,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=batch_count,
            continuation_token=token,
        )

        if not result:
            break

        for r in result:
            all_rows.append(
                {
                    "app_id": app_id,
                    "app_name": app_name,
                    "review_id": r.get("reviewId"),
                    "user_name": r.get("userName"),
                    "score": r.get("score"),
                    "content": r.get("content"),
                    "thumbs_up": r.get("thumbsUpCount"),
                    "at": r.get("at"),
                    "app_version": r.get("appVersion"),
                }
            )

        if token is None:
            break

    return all_rows

In [4]:
all_reviews = scrape_reviews_for_app(APP_ID, APP_NAME, target_count=TARGET_COUNT)

raw_df = pd.DataFrame(all_reviews)
print(f"\nTotal ulasan terkumpul (raw): {len(raw_df):,}".replace(",", "."))
raw_df.head()


Total ulasan terkumpul (raw): 100.000


,app_id,app_name,review_id,user_name,score,content,thumbs_up,at,app_version
0,com.levelinfinite.sgameGlobal,Honor of King,d335abc2-5966-40b3-b12a-d123d737d58e,Muhammad Barend,5,terima kasih Tencent udh mendengar kan playernya maaf yah kalau aku berkata kasar maaf yah soalnya sinyalnya ngaco M...,1,2026-04-27 10:01:43,11.3.1.9
1,com.levelinfinite.sgameGlobal,Honor of King,03a47e36-ab60-40ff-a989-80d6f617d57f,Iqbal Safii,5,👍🏻👍🏻👍🏻👍🏻,0,2026-04-27 09:40:17,NaN
2,com.levelinfinite.sgameGlobal,Honor of King,6843dcb2-a6ce-456c-ad39-56bf2eea7d65,Eko Purwoko Arbiansyah,5,game baik semoga makin sukses selalu dan ditingkatkan lagi performa nya,0,2026-04-27 09:38:00,11.3.1.9
3,com.levelinfinite.sgameGlobal,Honor of King,eec762a5-fae4-4f12-816f-39ba105f3f1d,Nur rohman Kamaru,2,gem tolol gajels,0,2026-04-27 09:33:50,NaN
4,com.levelinfinite.sgameGlobal,Honor of King,73704410-7065-49de-8824-dac309b09303,Teguh Hidayat,1,"Game sebenarnya seru dan grafiknya bagus, tapi sistemnya sangat mengecewakan. Saya main normal, tidak toxic, tidak A...",0,2026-04-27 09:31:57,11.3.1.9


In [5]:
df = raw_df.copy()

# Pembersihan dasar
before = len(df)
df = df.drop_duplicates(subset=["review_id"]).reset_index(drop=True)
df = df.dropna(subset=["content", "score"])
df["content"] = df["content"].astype(str).str.strip()
df = df[df["content"].str.len() > 3].reset_index(drop=True)

df["at"] = pd.to_datetime(df["at"], errors="coerce")

print(f"Sebelum cleaning : {before:,}".replace(",", "."))
print(f"Setelah cleaning : {len(df):,}".replace(",", "."))
print("Distribusi skor:")
display(df["score"].value_counts().sort_index())

# Simpan ke CSV
ordered_cols = [
    "review_id", "app_id", "app_name", "user_name", "score", "content", "thumbs_up", "at", "app_version"
]
existing_cols = [c for c in ordered_cols if c in df.columns]
df[existing_cols].to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nFile CSV tersimpan di: {OUTPUT_CSV.resolve()}")
print(f"Jumlah sampel akhir: {len(df):,}".replace(",", "."))

Sebelum cleaning : 100.000
Setelah cleaning : 97.817
Distribusi skor:


score
1    27577
2     4505
3     5420
4     7552
5    52763
Name: count, dtype: int64


File CSV tersimpan di: C:\Users\DianErdiana\dianerdiana-learning\machine-learning-dicoding\submission-deep-learning-01\data\reviews_hok.csv
Jumlah sampel akhir: 97.817


In [6]:
df.sample(5, random_state=42)[["app_name", "score", "content"]]

,app_name,score,content
20172,Honor of King,3,game nya bagus tapi kenapa mb nya kurangi lah mb nya sampe jadi gb memori ku penuh tolong mb nya 110 aja lah.
34580,Honor of King,5,mantap 👍🏻
38760,Honor of King,4,keren gemenya dan Gerafiny sukses selalu
79449,Honor of King,4,"Sistem Match making masib harus di perbaiki, rank saya sebelum restt itu di epic bintang 80 saat bermain rank saya b..."
30969,Honor of King,5,"hok tolong perbaiki sistem riport nya. banyak banget bocil bocil ml yang nggak jelas main, tolonglah perbaiki itu ag..."
